In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rc
from mpl_toolkits.mplot3d import Axes3D
rc('animation', html='jshtml')
from scipy.stats import  kstest
from itertools import product
from IPython.display import HTML
from concurrent.futures import ProcessPoolExecutor
import time

## Avances Algoritmos fraudulentos

En una primera instancia buscamos minimizar la funcion:
$$U(x)=\sum_{i=1}^m\sin^2(x_i)$$
Sobre el toro $m-$ dimensional, es decir, $(\mathbb{R}/ 2\pi\mathbb{Z})^m$

Trabajaremos asi con una variedad euclideana compacta y sin frontera. Parte importante de la funcion recae en que el minimo se alcanza con un valor 0, por lo que se usarán métodos estocasticos para aproximarse a él.

Una primera impresion sobre el conjunto a trabajar, nos indica los ceros de la funcion de forma concreta. En particular, dada la psoitividad de los terminos de $U(x)$ y la periodicidad que nos entrega el toro $m-dimensional$, los ceros de $U$ seran los posibles elementos $\hat{x}\in \mathbb{R}^m, x_i\in \{0,\pi\},\forall i=1,...,m$. En total tenemos $2^m$ ceros de $U$ en el toro.

Para obtener los minimos de la función realizaremos una aproximación estocastica hacia los puntos minimos de $U$. Como bien vimos en el curso, un método escencial al minimizar funciones es el de simmulating anneling. Para esta ocación, se utilizará un método similar que da ciertas ventajas al trabajar con funciones suaves.

El esquema para la minimización es el siguiente:

Dado un punto $x_0$ inicial, consideramos para una constante $h$ el paso dado por;
$$x_{n+1}=x_n-\beta\cdot \nabla U(x_n)\cdot h+\sqrt{2U(x_n)\cdot h}\cdot Y_n$$
El primer termino de resta corresponde a un descenso de gradiente sobre una funcion $U$, mientras que lo restante corresponde a un ajuste estocástico del tamaño del salto a medida que la función se minimiza. A medida que $U\to 0$, el segundo termino desaparece.

Lo importante de la construcción recae en que al hacer que $h\to 0$, el proceso discreto se vuelve "continuo" en su componente estocástica. Dado esto, consideraremos $h$ muy cercano a 0 en las simulaciones.

In [ ]:
#Funciones utiles

def U(x):
    return np.sum(np.sin(x)**2)

def nabla_U(x):
    return np.sin(2*x)

def dist_a_cero(x):
    r = x % np.pi
    return np.max(np.minimum(r, np.pi - r))

iteraciones_maximas=10**7
h=10**(-2)
error=10**(-8)

def SimulacionMinimos_sc(x_0,beta):
  m=len(x_0)
  assert beta>(m/2-1)
  x_0=np.array(x_0,dtype=float) % (2*np.pi)
  i=0
  while i<iteraciones_maximas:
    if dist_a_cero(x_0)<error:
      return x_0, i
    else:
      Y=np.random.randn(m)
      x_paso = x_0-beta*nabla_U(x_0)*h+np.sqrt(2*U(x_0)*h)*Y
      x_paso=x_paso % (2*np.pi)
      x_0=x_paso
      i+=1
  return x_0, None

# **Tasa de Convergencia**

Vemos en primer lugar los tiempos de llegada a los ceros (exponencial)

In [ ]:
num_vectores = 1000
m = 5
lista_x0 = np.random.uniform(0, 2*np.pi, size=(num_vectores,m))
tiempos=[]
for x in lista_x0:
  y,j = SimulacionMinimos_sc(x,2)
  tiempos.append(j*h)
plt.hist(tiempos, bins=50, color='indigo', edgecolor='black')
plt.show()

La idea es ver la tasa de convergencia de la distancia entre su proceso y el valor final. Para eso realizamos simulaciones verificamos si es que el valor de la distancia decae

In [ ]:
iteraciones_maximas = 50000
h = 10**(-2)
def simular_trayectoria(x_0, beta):
    m = len(x_0)
    x_0 = np.array(x_0, dtype=float) % (2*np.pi)
    tiempos = []
    distancias = []

    for i in range(iteraciones_maximas):
        d = dist_a_cero(x_0)
        tiempos.append(i * h)
        distancias.append(d)

        if d < 10**-8:
            break

        Y = np.random.randn(m)
        x_paso = x_0 - beta * nabla_U(x_0) * h + np.sqrt(2 * U(x_0) * h) * Y
        x_0 = x_paso % (2 * np.pi)

    return np.array(tiempos), np.array(distancias)

# Parámetros del experimento
m = 5
# Fijamos un punto inicial aleatorio para comparar justamente con los mismos valores
np.random.seed(42)
x_inicial = np.random.uniform(0.5, 2.5, size=m)

# Valores de beta supercríticos (para m=5, m/2 - 1 = 1.5, por lo que beta > 1.5)
betas = [1.8, 2.2, 2.8, 3.5]
pendientes = []
plt.figure(figsize=(10, 6))

for beta in betas:
    t, d = simular_trayectoria(x_inicial, beta)
    # Evitamos ceros exactos para poder aplicar logaritmo
    d_safe = np.where(d <= 0, 1e-16, d)

    # Graficamos el logaritmo de la distancia vs el tiempo
    plt.plot(t, np.log(d_safe), label=f"Beta = {beta}")
plt.xlabel("Tiempo (t)")
plt.ylabel("Log(Distancia al cero)")
plt.title("Tasa de convergencia del proceso para diferentes valores de Beta")
plt.legend()
plt.grid(True)
plt.show()

betas = [1.8, 2.2, 2.8, 3.5]
pendientes = []

x_inicial = np.random.uniform(0.5, 2.5, size=5)

for beta in betas:
    t, d = simular_trayectoria(x_inicial, beta)
    d_safe = np.where(d <= 0, 1e-16, d)
    log_d = np.log(d_safe)
    mask = (log_d < 0) & (log_d > -15)
    if np.sum(mask) > 10:
        pendiente, _ = np.polyfit(t[mask], log_d[mask], 1)
        pendientes.append(abs(pendiente))
    else:
        pendientes.append(0)

# Gráfico de la tasa de convergencia vs Beta
plt.figure(figsize=(8, 5))
plt.plot(betas, pendientes, 'o-', color='purple', linewidth=2, markersize=8)
plt.xlabel("Valor de Beta")
plt.ylabel("Tasa de Convergencia (Magnitud de la pendiente)")
plt.title("Relación lineal entre Beta y la tasa de convergencia")
plt.grid(True)
plt.show()

Notemos que a medida que el β aumenta, la curva se ve mucho más pronunciada. Por ejemplo, para β= 3.5 la curva cae mucho más rápido, mientras que para β = 1.8 se demora más en caer la curva. Entonces, gracias al gráfico se puede visualizar que la rapidez en que decae la distancia es directamente proporcional al valor de β.

Además, en el segundo gráfico se verifica que la relación entre estos valores es lineal, lo cual no se logra ver muy bien en el primer gráfico, que tiene una escala logaritmica, si bien las curvas no se ven como rectas, el comportamiento general de esta si está siendo "lineal".

# **Función con peso**
Ahora consideremos la siguiente función $$U(x)=\sum_{i=1}^mc_i\sin^2(x_i)$$, y se realiza lo mismo que para la función inicial.

In [ ]:
def U_conpeso(x,c):
  return np.sum(c*np.sin(x)**2)

def nabla_U_conpeso(x,c):
  return c*np.sin(2*x)

iteraciones_maximas=10**7
h=10**(-2)
error=10**(-8)

def SimulacionMinimos_sc_conpeso(x_0, beta, c):
    m = len(x_0)
    assert beta > (m/2 - 1)
    x_0 = np.array(x_0, dtype=float) % (2 * np.pi)
    i = 0
    while i < iteraciones_maximas:
        if dist_a_cero(x_0) < error:
            return x_0, i
        else:
            Y = np.random.randn(m)
            u_val = np.sum(c * np.sin(x_0)**2)
            grad_val = c * np.sin(2 * x_0)
            u_val = max(0.0, u_val)
            x_paso = x_0 - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
            x_paso = x_paso % (2 * np.pi)
            x_0 = x_paso
            i += 1
    return x_0, None


Tiempo de llegada a ceros (exponencial)

In [ ]:
num_vectores = 1000
m = 5
pesos = np.array([0.5, 1.0, 1.5, 2.0, 2.5])
lista_x0 = np.random.uniform(0, 2*np.pi, size=(num_vectores,m))
tiempos=[]
for x in lista_x0:
  y,j = SimulacionMinimos_sc_conpeso(x,2,c=pesos)
  if j is not None:
        tiempos.append(j * h)
plt.hist(tiempos, bins=50, color='indigo', edgecolor='black')
plt.show()

In [ ]:
iteraciones_maximas=50000
h=10**(-2)
def simular_trayectoria_conpeso(x_0, beta, c):
    m = len(x_0)
    x_0 = np.array(x_0, dtype=float) % (2 * np.pi)
    tiempos = []
    distancias = []

    for i in range(iteraciones_maximas):
        d = dist_a_cero(x_0)
        tiempos.append(i * h)
        distancias.append(d)

        if d < 10**-8:
            break

        Y = np.random.randn(m)
        u_val = U_conpeso(x_0, c)
        grad_val = nabla_U_conpeso(x_0, c)
        u_val = max(0.0, u_val)
        x_paso = x_0 - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
        x_0 = x_paso % (2 * np.pi)

    return np.array(tiempos), np.array(distancias)

#--------------------------------------------------------------------------#
m = 5
np.random.seed(42)
pesos = np.array([0.5, 1.0, 1.5, 2.0, 2.5])
x_inicial = np.random.uniform(0.5, 2.5, size=m)
betas = [1.8, 2.2, 2.8, 3.5]
plt.figure(figsize=(10, 6))

for beta in betas:
    t, d = simular_trayectoria_conpeso(x_inicial, beta, pesos)
    d_safe = np.where(d <= 0, 1e-16, d)
    plt.plot(t, np.log(d_safe), label=f"Beta = {beta} (con pesos)")

plt.xlabel("Tiempo (t)")
plt.ylabel("Log(Distancia al cero)")
plt.title(f"Tasa de convergencia con función ponderada (Pesos: {pesos})")
plt.legend()
plt.grid(True)
plt.show()

Y se cumple lo mismo sobre la relación de β y la convergencia, son directamente proporcionales.

# **Visualizacion del algoritmo para el toro con m =1, 2**

Convergencia de $x_0$ en el plano $T^2$

In [ ]:
m = 2
beta = 2.0
h = 0.01
n_pasos = 600

np.random.seed(0)
x = np.array([1.0, 5.0])
trayectoria = [x.copy()]

for _ in range(n_pasos):
    Y = np.random.randn(m)
    u_val = np.sum(np.sin(x)**2)
    grad_val = np.sin(2 * x)
    u_val = max(0.0, u_val)
    x_paso = x - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
    x = x_paso % (2 * np.pi)
    trayectoria.append(x.copy())

trayectoria = np.array(trayectoria)


fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(0, 2 * np.pi)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Caminata estocástica sobre el Toro (m=2)')

minimos = [0, np.pi, 2*np.pi]
for mx in minimos:
    for my in minimos:
        ax.plot(mx, my, 'ro', markersize=8, alpha=0.6)
linea_trayectoria, = ax.plot([], [], lw=1.5, color='indigo', alpha=0.7, label='Trayectoria')
punto_actual, = ax.plot([], [], 'bo', markersize=8, label='Posición actual $X_t$')
ax.legend(loc='upper right')

def init():
    linea_trayectoria.set_data([], [])
    punto_actual.set_data([], [])
    return linea_trayectoria, punto_actual

def update(frame):

    x_vals = trayectoria[:frame, 0]
    y_vals = trayectoria[:frame, 1]
    linea_trayectoria.set_data(x_vals, y_vals)

    pos_x = [trayectoria[frame, 0]]
    pos_y = [trayectoria[frame, 1]]
    punto_actual.set_data(pos_x, pos_y)
    return linea_trayectoria, punto_actual

ani = FuncAnimation(fig, update, frames=len(trayectoria), init_func=init, interval=30, blit=True)
ani

Convergencia de $x_0$ en el toroide (3D)

In [ ]:
m = 2
beta = 2.0
h = 0.01
n_pasos = 400

np.random.seed(42)
x_2d = np.array([1.0, 5.0])
trayectoria_2d = [x_2d.copy()]

for _ in range(n_pasos):
    Y = np.random.randn(m)
    u_val = np.sum(np.sin(x_2d)**2)
    grad_val = np.sin(2 * x_2d)
    u_val = max(0.0, u_val)

    x_2d_paso = x_2d - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
    x_2d = x_2d_paso % (2 * np.pi)
    trayectoria_2d.append(x_2d.copy())

trayectoria_2d = np.array(trayectoria_2d)

#------------------------------IA--------------------------------------------#

def mapear_a_3d(x_2d_array, R_major=3.0, r_minor=1.0):
    theta = x_2d_array[:, 0]  # x1 -> ángulo grande
    phi = x_2d_array[:, 1]    # x2 -> ángulo pequeño

    # Ecuaciones paramétricas estándar del toro 3D
    X_3d = (R_major + r_minor * np.cos(phi)) * np.cos(theta)
    Y_3d = (R_major + r_minor * np.cos(phi)) * np.sin(theta)
    Z_3d = r_minor * np.sin(phi)

    return X_3d, Y_3d, Z_3d

# Convertimos toda la trayectoria plana a 3D
trayectoria_3d_X, trayectoria_3d_Y, trayectoria_3d_Z = mapear_a_3d(trayectoria_2d)

# --- 3. CONFIGURACIÓN DE LA ANIMACIÓN 3D ---
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

# Parámetros estéticos del toro base
R = 3.0
r = 1.0
U_surf = np.linspace(0, 2 * np.pi, 50)
V_surf = np.linspace(0, 2 * np.pi, 50)
U_surf, V_surf = np.meshgrid(U_surf, V_surf)
X_surf = (R + r * np.cos(V_surf)) * np.cos(U_surf)
Y_surf = (R + r * np.cos(V_surf)) * np.sin(U_surf)
Z_surf = r * np.sin(V_surf)

# Dibujar la dona de fondo (estática)
ax.plot_surface(X_surf, Y_surf, Z_surf, cmap='viridis', alpha=0.1, edgecolor='none')

# Dibujar los mínimos teóricos (0, pi, 2pi envueltos)
# En m=2 plana hay 4: (0,0), (pi,0), (0,pi), (pi,pi)
minimos_planos = np.array([[0,0], [np.pi, 0], [0, np.pi], [np.pi, np.pi]])
min_X, min_Y, min_Z = mapear_a_3d(minimos_planos, R, r)
ax.scatter(min_X, min_Y, min_Z, color='red', s=100, alpha=0.8, label='Mínimos')

# Elementos animados
linea_3d, = ax.plot([], [], [], lw=1.0, color='indigo', alpha=0.6)
punto_3d, = ax.plot([], [], [], 'bo', markersize=8, label='Posición $X_t$')

# Ajustes de vista y ejes
ax.set_xlim(-R-r, R+r)
ax.set_ylim(-R-r, R+r)
ax.set_zlim(-R-r, R+r)
ax.set_title('Trayectoria Estocástica Envuelta en el Toro 3D')
ax.legend()
ax.view_init(elev=30, azim=45) # Ángulo de cámara inicial
ax.set_axis_off() # Ocultar ejes para mejor estética

def update(frame):
    # Rastro acumulado (puedes ajustar cuántos pasos mostrar para no saturar)
    pasos_rastro = max(0, frame - 100)
    linea_3d.set_data(trayectoria_3d_X[pasos_rastro:frame], trayectoria_3d_Y[pasos_rastro:frame])
    linea_3d.set_3d_properties(trayectoria_3d_Z[pasos_rastro:frame])

    # Punto actual
    punto_3d.set_data([trayectoria_3d_X[frame]], [trayectoria_3d_Y[frame]])
    punto_3d.set_3d_properties([trayectoria_3d_Z[frame]])

    # Pequeña rotación de cámara para ver mejor
    ax.view_init(elev=30, azim=45 + frame * 0.1)

    return linea_3d, punto_3d

ani = FuncAnimation(fig, update, frames=len(trayectoria_2d), interval=30, blit=False)
ani

Convergencia de $x_0$ en la circunferencia

In [ ]:
m = 1
beta = 1.0
h = 0.01
n_pasos = 400

np.random.seed(42)
x = np.array([2.0])
trayectoria = [x.copy()]

for _ in range(n_pasos):
    Y = np.random.randn(m)
    u_val = np.sum(np.sin(x)**2)
    grad_val = np.sin(2 * x)
    u_val = max(0.0, u_val)

    x_paso = x - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
    x = x_paso % (2 * np.pi)
    trayectoria.append(x.copy())

trayectoria = np.array(trayectoria)

#------------------------------IA-----------------------------------------#

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location("E")  # El 0 inicia a la derecha
ax.set_theta_direction(1)        # Dirección antihoraria

# Establecer límites radiales para que el círculo quede limpio
ax.set_ylim(0, 1.5)
ax.yaxis.set_visible(False)      # Ocultar marcas radiales

ax.set_title('Caminata estocástica en el Toro 1D', va='top')

# Dibujar los mínimos teóricos (en coordenadas polares: ángulo = x, radio = 1.0)
minimos_angulos = [0, np.pi, 2 * np.pi]
ax.plot(minimos_angulos, [1.0] * len(minimos_angulos), 'ro', markersize=10, alpha=0.7, label='Mínimos (Ceros)')

# Elementos animados en polares (theta, r)
punto_actual, = ax.plot([], [], 'bo', markersize=10, label='Posición $X_t$')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

def init():
    punto_actual.set_data([], [])
    return punto_actual,

def update(frame):
    # El ángulo es la posición x del toro, y el radio se fija en 1.0
    angulo = trayectoria[frame, 0]
    punto_actual.set_data([angulo], [1.0])
    return punto_actual,

ani = FuncAnimation(fig, update, frames=len(trayectoria), init_func=init, interval=30, blit=True)
ani

Convergencia de 10 puntos sobre la circunferencia

In [ ]:
n_puntos = 10
beta = 1.0
h = 0.01
n_pasos = 400

np.random.seed(42)
x = np.random.uniform(0, 2 * np.pi, size=n_puntos)
trayectoria = np.zeros((n_pasos, n_puntos))
trayectoria[0] = x
for t in range(1, n_pasos):
    Y = np.random.randn(n_puntos)
    u_val = np.sin(x)**2
    grad_val = np.sin(2 * x)
    u_val = np.maximum(0.0, u_val)
    x_paso = x - beta * grad_val * h + np.sqrt(2 * u_val * h) * Y
    x = x_paso % (2 * np.pi)
    trayectoria[t] = x

#-------------------------------IA------------------------------------------#
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location("E")  # El ángulo 0 empieza a la derecha
ax.set_theta_direction(1)        # Sentido antihorario

ax.set_ylim(0, 1.5)
ax.yaxis.set_visible(False)      # Ocultar marcas del radio

ax.set_title('Caminata estocástica de 10 puntos en el Círculo (Toro 1D)', va='top')

# Dibujar los mínimos teóricos (ceros del potencial en θ = 0, π, 2π)
minimos_angulos = [0, np.pi, 2 * np.pi]
ax.plot(minimos_angulos, [1.0] * len(minimos_angulos), 'ro', markersize=10, alpha=0.7, label='Mínimos (Ceros)')

# Elemento animado: un scatter plot que maneja las posiciones de los 10 puntos (radio r = 1.0)
puntos_actuales = ax.scatter(trayectoria[0], np.ones(n_puntos), c='blue', s=80, label='Posiciones $X_t$')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

def update(frame):
    # En cada cuadro actualizamos los ángulos theta de los 10 puntos
    angulos = trayectoria[frame]
    # np.c_ combina los ángulos y el radio constante 1.0 para los 10 puntos
    puntos_actuales.set_offsets(np.c_[angulos, np.ones(n_puntos)])
    return puntos_actuales,

ani = FuncAnimation(fig, update, frames=n_pasos, interval=30, blit=True)
ani

Animacion Nítida

In [ ]:
#-----------------------------IA--------------------------------------------#
fig, ax = plt.subplots(figsize=(6, 6))

ax.set_aspect('equal')
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)

ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8)

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)


theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta),
        color='black', linewidth=1.2)

# Mínimos teóricos: theta = 0 y theta = pi
minimos_x = [1, -1]
minimos_y = [0, 0]

ax.scatter(
    minimos_x, minimos_y,
    c='red', s=100, zorder=3
)

# Puntos móviles
x_cart = np.cos(trayectoria[0])
y_cart = np.sin(trayectoria[0])

puntos_actuales = ax.scatter(
    x_cart, y_cart,
    c='blue', s=80, zorder=2
)

# Actualización
def update(frame):
    angulos = trayectoria[frame]

    posiciones_x = np.cos(angulos)
    posiciones_y = np.sin(angulos)

    puntos_actuales.set_offsets(
        np.c_[posiciones_x, posiciones_y]
    )

    return puntos_actuales,

ani = FuncAnimation(
    fig, update,
    frames=n_pasos,
    interval=30,
    blit=True
)
ani

# **Estudio de $\beta$ subcrítico**

In [ ]:
import numpy as np, matplotlib.pyplot as plt

m, beta = 2, -0.5
h, T, N = 1e-3, 20.0, 200
nb = 20
cada = 100

M = 2000
g = (np.arange(M) + 0.5)*2*np.pi/M
G1, G2 = np.meshgrid(g, g, indexing='ij')
w = (np.sin(G1)**2 + np.sin(G2)**2)**(-1 - beta)
w /= w.sum()
bordes = np.linspace(0, 2*np.pi, nb+1)
q = np.histogram2d(G1.ravel(), G2.ravel(), bins=[bordes, bordes], weights=w.ravel())[0]
q /= q.sum()


rng = np.random.default_rng(0)
piso = np.mean([0.5*np.abs(rng.multinomial(N, q.ravel()).reshape(nb, nb)/N - q).sum()
                for _ in range(20)])


def paso(X):
    Z = np.random.randn(*X.shape)
    s = np.sqrt(2*np.sum(np.sin(X)**2, axis=1)*h)[:, None]
    return (X - beta*np.sin(2*X)*h + s*Z) % (2*np.pi)

X = np.full((N, m), np.pi/2)
tv, tt = [], []
for n in range(int(T/h) + 1):
    if n % cada == 0:
        p = np.histogram2d(X[:, 0], X[:, 1], bins=[bordes, bordes])[0]/N
        tv.append(0.5*np.abs(p - q).sum()); tt.append(n*h)
    X = paso(X)

plt.plot(tt, tv, label='$d_{VT}$ (ley empírica, $\\pi_\\beta$)')
plt.axhline(piso, color='r', ls='--')
plt.xlabel('t'); plt.legend(); plt.show()

Trayectorias para $\beta$ subcrítico

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter

m, beta = 2, -0.5
h, T, P, eps = 1e-3, 30.0, 2, 0.3
pasos = int(T/h); cada = 50; K = 40

U  = lambda X: np.sum(np.sin(X)**2, axis=-1)
dU = lambda X: np.sin(2*X)


#--------------------------------IA---------------------------------------#
def d_ceros(X):
    r = X % np.pi
    return np.linalg.norm(np.minimum(r, np.pi - r), axis=-1)

rng = np.random.default_rng(1)
X = rng.uniform(0, 2*np.pi, size=(P, m))
tray = np.empty((pasos+1, P, m)); tray[0] = X
cont = np.zeros((pasos+1, P), dtype=int)
dentro = d_ceros(X) < eps
cont[0] = dentro
for n in range(pasos):
    Z = rng.standard_normal((P, m))
    X = (X - beta*dU(X)*h + np.sqrt(2*U(X)*h)[:, None]*Z) % (2*np.pi)
    d = d_ceros(X)
    entra = (d < eps) & ~dentro
    dentro = np.where(entra, True, np.where(d > 2*eps, False, dentro))
    cont[n+1] = cont[n] + entra
    tray[n+1] = X
dist = d_ceros(tray)
t = np.arange(pasos+1)*h

colores = ['tab:blue', 'tab:orange']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [1, 1.3]})
ax1.set_xlim(0, 2*np.pi); ax1.set_ylim(0, 2*np.pi); ax1.set_aspect('equal')
for cx in (0, np.pi, 2*np.pi):
    for cy in (0, np.pi, 2*np.pi):
        ax1.add_patch(plt.Circle((cx, cy), eps, fill=False, ls='--', color='gray'))
        ax1.plot(cx, cy, 'kx')
estelas = [ax1.scatter([], [], s=10, alpha=.35, color=c) for c in colores[:P]]
cabezas = [ax1.scatter([], [], s=70, color=c, edgecolor='k', zorder=5) for c in colores[:P]]

ax2.set_xlim(0, T); ax2.set_ylim(1e-4, 3); ax2.set_yscale('log')
ax2.axhline(eps, color='gray', ls='--')
lineas = [ax2.plot([], [], color=c, lw=.8, label=f'partícula {i+1}')[0] for i, c in enumerate(colores[:P])]
ax2.set_xlabel('t'); ax2.set_ylabel('distancia al conjunto de ceros'); ax2.legend(loc='lower left')

def actualizar(k):
    s = k*cada
    a = max(0, s - (K-1)*cada)
    for j in range(P):
        estelas[j].set_offsets(tray[a:s+1:cada, j, :])
        cabezas[j].set_offsets(tray[s:s+1, j, :])
        lineas[j].set_data(t[:s+1:cada], dist[:s+1:cada, j])
    ax1.set_title(f't = {t[s]:.1f}   visitas: ' + ' | '.join(f'p{j+1}={cont[s, j]}' for j in range(P)))

ani = FuncAnimation(fig, actualizar, frames=pasos//cada + 1, interval=40)
plt.close(fig)
writer = FFMpegWriter(fps=25, metadata={'artist': 'Yo'}, bitrate=2000)
ani.save('trayectorias.mp4', writer=writer)
HTML(ani.to_jshtml())

Mapa de calor para la medida invariante

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from math import sin, sqrt

m, beta = 2, -0.5              # beta_0 = 0 -> subcrítico
h, T = 1e-3, 3000.0
n = int(T/h); tau = 2*np.pi
nb, eps = 20, 0.3

# --- UNA trayectoria larga ---
rng = np.random.default_rng(0)
Z = rng.standard_normal((n, 2))
tray = np.empty((n, 2))
x1, x2 = rng.uniform(0, tau, 2)
for i in range(n):
    s = sqrt(2*(sin(x1)**2 + sin(x2)**2)*h)
    x1 = (x1 - beta*sin(2*x1)*h + s*Z[i, 0]) % tau
    x2 = (x2 - beta*sin(2*x2)*h + s*Z[i, 1]) % tau
    tray[i] = (x1, x2)
tray = tray[n//20:]            # descarta el 5% inicial

# --- medida teórica: masa por bin y masa cerca de los ceros ---
def d_ceros(X):
    r = X % np.pi
    return np.linalg.norm(np.minimum(r, np.pi - r), axis=-1)

M = 2000
g = (np.arange(M) + 0.5)*tau/M
G1, G2 = np.meshgrid(g, g, indexing='ij')
w = (np.sin(G1)**2 + np.sin(G2)**2)**(-1 - beta); w /= w.sum()
bordes = np.linspace(0, tau, nb+1)
q = np.histogram2d(G1.ravel(), G2.ravel(), bins=[bordes, bordes], weights=w.ravel())[0]
q /= q.sum()
masa_teo = w[d_ceros(np.stack([G1, G2], -1)) < eps].sum()

# --- medida empírica de esa trayectoria ---
p = np.histogram2d(tray[:, 0], tray[:, 1], bins=[bordes, bordes])[0]; p /= p.sum()
frac_emp = np.mean(d_ceros(tray) < eps)
print(f"fracción de tiempo a distancia < {eps} de los ceros: empírica {frac_emp:.4f}, teórica {masa_teo:.4f}")

# (1) mapas de calor: empírico, teórico, diferencia
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
vmax = max(p.max(), q.max())
for a, Z_, tit in zip(ax[:2], (p, q), ('trayectoria (tiempo)', 'teórica $\\pi_\\beta$')):
    im = a.imshow(Z_.T, origin='lower', extent=[0, tau, 0, tau], vmax=vmax); a.set_title(tit)
im = ax[2].imshow((p - q).T, origin='lower', extent=[0, tau, 0, tau], cmap='coolwarm',
                  vmin=-vmax/3, vmax=vmax/3); ax[2].set_title('diferencia')
plt.show()

# (2) la TV debe bajar con el tiempo de observación, ~ T^(-1/2)
puntos = np.unique(np.geomspace(1000, len(tray), 25).astype(int))
tv = [0.5*np.abs(np.histogram2d(tray[:k, 0], tray[:k, 1], bins=[bordes, bordes])[0]/k - q).sum()
      for k in puntos]
plt.loglog(puntos*h, tv, 'o-', label='TV')
plt.loglog(puntos*h, tv[0]*np.sqrt(puntos[0]/puntos), 'r--', label='pendiente -1/2')
plt.xlabel('tiempo de observación'); plt.legend(); plt.show()

# **Estudio de medida sobre $C$ para $\beta$ supercrítico**

Ahora utilizando la función:
$$U(x)=f(x) \;=\; \sum_{k \in K}  a_k \cos (k \cdot x) + b_k \sin (k\cdot x), \qquad x \in T^m$$

Con $K:= \{ k \in \mathbb{Z}^m : 0 < |k| \leq \kappa \}, a_k, b_k \sim N(0,\sigma_k^2)$

In [ ]:
def conjunto_K(m, kappa):
  vectores = []
  rango = range(-kappa,kappa+1)
  for k in product(rango,repeat=m):
    k_arr = np.array(k, dtype=float)
    norma = np.linalg.norm(k_arr)
    if 0 < norma<=kappa:
      vectores.append(k_arr)
  return vectores

def coeficientes(vectores, sigma_k=1):
  a = {}
  b = {}
  for k in vectores:
    a[tuple(k)] = np.random.normal(0, sigma_k)
    b[tuple(k)] = np.random.normal(0, sigma_k)
  return a,b


def U2(x, vectores, a, b):
    K_mat = np.array(vectores)
    prods = np.dot(K_mat, x)
    a_vals = np.array([a[tuple(k)] for k in vectores])
    b_vals = np.array([b[tuple(k)] for k in vectores])
    return np.sum(a_vals * np.cos(prods) + b_vals * np.sin(prods))

def nabla_U2(x, k_vectores, a, b):
    K_mat = np.array(k_vectores)
    prods = np.dot(K_mat, x)
    a_vals = np.array([a[tuple(k)] for k in k_vectores])
    b_vals = np.array([b[tuple(k)] for k in k_vectores])
    deriv_terms = -a_vals * np.sin(prods) + b_vals * np.cos(prods)

    return np.dot(deriv_terms, K_mat)

La idea ahora es evaluar cuantos ceros tiene la función utilizando diferentes $κ \in [2,6]$

In [ ]:
def estimar_ceros(m, kappa, num_trayectorias=500, beta=2.0):
    vectores = conjunto_K(m, kappa)
    a, b = coeficientes(vectores)

    puntos_finales = []

    for _ in range(num_trayectorias):
        x_0 = np.random.uniform(0, 2 * np.pi, size=m)
        x = x_0.copy()
        h = 1e-2
        for _ in range(20000):
            u_val = U2(x, vectores, a, b)
            u_val = max(0.0, u_val)
            grad = nabla_U2(x, vectores, a, b)
            Y = np.random.randn(m)
            x_paso = x - beta * grad * h + np.sqrt(2 * u_val * h) * Y
            x = x_paso % (2 * np.pi)
        puntos_finales.append(x)
    puntos_finales = np.array(puntos_finales)
    puntos_redondeados = np.round(puntos_finales, decimals=2)
    ceros_unicos = np.unique(puntos_redondeados, axis=0)
    return len(ceros_unicos), vectores

def _simular_una_trayectoria(args):
  m, kappa, beta, vectores, a, b = args
  x_0 = np.random.uniform(0, 2 * np.pi, size=m)
  x = x_0.copy()
  h = 1e-2
  for _ in range(3000):
    u_val = U2(x, vectores, a, b)
    u_val = max(0.0, u_val)
    grad = nabla_U2(x, vectores, a, b)
    Y = np.random.randn(m)
    x_paso = x - beta * grad * h + np.sqrt(2 * u_val * h) * Y
    x = x_paso % (2 * np.pi)
  return x

#----------------------------IA-----------------------------------------#

def estimar_ceros_rapido(m, kappa, num_trayectorias=100, beta=2.0):
  vectores = conjunto_K(m, kappa)
  a, b = coeficientes(vectores)

  # Empaquetamos los argumentos para cada proceso
  args_list = [(m, kappa, beta, vectores, a, b) for _ in range(num_trayectorias)]

  # Usamos todos los núcleos disponibles de tu CPU
  with ProcessPoolExecutor() as executor:
    puntos_finales = list(executor.map(_simular_una_trayectoria, args_list))

  puntos_finales = np.array(puntos_finales)

  puntos_redondeados = np.round(puntos_finales, decimals=2)
  ceros_unicos = np.unique(puntos_redondeados, axis=0)

  return len(ceros_unicos), vectores, puntos_finales


# --- EJECUCIÓN Y GRÁFICO CUADRADO ---
m = 2
kappa = 4

num_ceros, vectores, puntos = estimar_ceros_rapido(
    m, kappa, num_trayectorias=100, beta=2.0
)

# Generación del gráfico cuadrado exacto para el toroide [0, 2pi] x [0, 2pi]
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    puntos[:, 0],
    puntos[:, 1],
    alpha=0.6,
    s=35,
    color='crimson',
    edgecolors='none',
    label='Trayectorias convergentes',
)

# Configuración estricta de un gráfico cuadrado
ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(0, 2 * np.pi)
ax.set_aspect('equal', adjustable='box')

ax.set_xlabel(r'$x_1$', fontsize=12)
ax.set_ylabel(r'$x_2$', fontsize=12)
ax.set_title(
    f'Toroide $m=2, \\kappa={kappa}$\nCeros únicos detectados: {num_ceros}',
    fontsize=13,
)

# Marcas de los ejes en múltiplos de pi
ticks = [0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi]
tick_labels = ['$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$']
ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels)
ax.set_yticks(ticks)
ax.set_yticklabels(tick_labels)
ax.grid(True, linestyle='--', alpha=0.5)

ax.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

Para estudiar la distribuión de la medida invariante, en el beta supercrítico (β>m/2−1), el proceso pasa una gran cantidad de tiempo explorando y acumulándose cerca de los mínimos de la función. El objetivo aquí es estudiar qué tanto se distancia esta medida empírica de la medida uniforme en el toro al variar κ.

In [ ]:
def estudiar_medida_supercritica(m, kappa, beta=2.0, pasos=30000):
    vectores = conjunto_K(m, kappa)
    a, b = coeficientes(vectores)

    x = np.random.uniform(0, 2 * np.pi, size=m)
    h = 1e-2
    trayectoria_muestras = []

    for step in range(pasos):
        u_val = U2(x, vectores, a, b)
        u_val = max(0.0, u_val)
        grad = nabla_U2(x, vectores, a, b)
        Y = np.random.randn(m)
        x_paso = x - beta * grad * h + np.sqrt(2 * u_val * h) * Y
        x = x_paso % (2 * np.pi)
        if step % 10 == 0:
            trayectoria_muestras.append(x.copy())

    trayectoria_muestras = np.array(trayectoria_muestras)

    estadisticas_ks = []
    for dim in range(m):
        datos_dim = trayectoria_muestras[:, dim]
        # Test KS contra Uniform(0, 2*pi)
        stat, p_value = kstest(datos_dim, 'uniform', args=(0, 2 * np.pi))
        estadisticas_ks.append(stat)
    return np.mean(estadisticas_ks)


distancias_ks = []
for kap in list(range(2, 9)):
    distancia_promedio = estudiar_medida_supercritica(m=2, kappa=kap, beta=2.0)
    print(f"Kappa = {kap}: Estadístico KS promedio (distancia a la uniforme) = {distancia_promedio:.4f}")
    distancias_ks.append(distancia_promedio)
#------------------------------IA------------------------------------#
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    list(range(2, 9)),
    distancias_ks,
    marker='o',
    linestyle='-',
    color='tab:blue',
    linewidth=2.5,
    markersize=8,
    label=r'Estadístico KS promedio ($\beta = 2.0$)',
)
ax.set_title(
    r'Distancia de la uniforme segun variaciones de $\kappa$',
    fontsize=13,
    pad=12,
)
ax.set_ylabel('Estadístico KS', fontsize=12)
ax.set_xticks(list(range(2, 9)))
ax.grid(True, linestyle='--', alpha=0.6)
ax.axhline(0, color='gray', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

Ejemplo gráfico:

In [ ]:
m = 2
kappa = 3
beta = 2.5
h = 0.005
n_particulas = 400
total_steps = 2000
num_frames = 120

k_vec = conjunto_K(m, kappa)
a, b = coeficientes(k_vec)


def f_y_grad_f(x, k_vectores, a, b):
  f_val = 0.0
  grad_f = np.zeros_like(x)
  for k in k_vectores:
    kt = tuple(k)
    prod = np.dot(k, x)
    c, s = np.cos(prod), np.sin(prod)
    f_val += a[kt] * c + b[kt] * s
    grad_f += k * (-a[kt] * s + b[kt] * c)
  return f_val, grad_f





#--------------------------IA-----------------------------------------#
raw_steps = np.linspace(0, np.sqrt(total_steps), num_frames)
step_schedule = np.unique(
    (raw_steps**2).astype(int)
)  # Índices exactos de pasos a guardar
actual_frames = len(step_schedule)

# Inicializamos partículas uniformemente en todo el toroide
particulas = np.random.uniform(0, 2 * np.pi, size=(n_particulas, m))

# Matriz para almacenar las posiciones en cada fotograma de la animación
trayectoria_anim = np.zeros((actual_frames, n_particulas, m))
trayectoria_anim[0] = particulas.copy()

print('Simulando y precalculando estados...')
current_step = 0
for f_idx in range(1, actual_frames):
  target_step = step_schedule[f_idx]
  steps_to_run = target_step - current_step

  for _ in range(steps_to_run):
    for p in range(n_particulas):
      f_val, grad_f = f_y_grad_f(particulas[p], k_vec, a, b)
      u_val = f_val**2
      grad_u = (
          2 * f_val * grad_f
      )  # Regla de la cadena: nabla(f^2) = 2*f*nabla(f)
      ruido = np.random.randn(m)
      particulas[p] = (
          particulas[p] - beta * grad_u * h + np.sqrt(2 * u_val * h) * ruido
      ) % (2 * np.pi)
    current_step += 1

  trayectoria_anim[f_idx] = particulas.copy()

#--------------------------IA-----------------------------------------#
grid_res = 100
x_vals = np.linspace(0, 2 * np.pi, grid_res)
y_vals = np.linspace(0, 2 * np.pi, grid_res)
X, Y = np.meshgrid(x_vals, y_vals)
Z = np.zeros_like(X)

for i in range(grid_res):
  for j in range(grid_res):
    f_v, _ = f_y_grad_f(np.array([X[i, j], Y[i, j]]), k_vec, a, b)
    Z[i, j] = f_v**2

#--------------------------IA-----------------------------------------#
fig, ax = plt.subplots(figsize=(7, 6))
contour = ax.contourf(X, Y, np.log1p(Z), levels=30, cmap='viridis')
plt.colorbar(contour, label='log(1 + f(x)²) [Valles oscuros]')

scatter = ax.scatter(
    trayectoria_anim[0, :, 0],
    trayectoria_anim[0, :, 1],
    color='red',
    edgecolor='white',
    s=25,
    alpha=0.8,
    label='Partículas X(t)',
)

ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(0, 2 * np.pi)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title(
    f'Descenso de gradiente estocastico (κ={kappa}, β={beta})'
)
ax.legend(loc='upper right')


def actualizar(frame):
  # Actualizamos las posiciones con los datos precalculados
  scatter.set_offsets(trayectoria_anim[frame])
  paso_actual = step_schedule[frame]
  return scatter,

anim = FuncAnimation(
    fig, actualizar, frames=actual_frames, interval=50, blit=True
)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
fig = plt.figure(figsize=(14, 6))
kappas_a_comparar = [2, 5]

for idx, kap in enumerate(kappas_a_comparar):
    ax = fig.add_subplot(1, 2, idx + 1, projection='3d')

    k_vec = conjunto_K(2, kap)
    a, b = coeficientes(k_vec)

    Z_kap = np.zeros_like(X)
    for i in range(grid_res):
        for j in range(grid_res):
            f_v, _ = f_y_grad_f(np.array([X[i,j], Y[i,j]]), k_vec, a, b)
            Z_kap[i,j] = f_v**2

    surf = ax.plot_surface(X, Y, Z_kap, cmap='magma', edgecolor='none', alpha=0.9)
    ax.set_title(f'Paisaje de U(x) = f(x)² con κ = {kap}\n(Frecuencias en K: {len(k_vec)})')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_zlabel('U(x)')
    ax.view_init(elev=35, azim=45)

plt.tight_layout()
plt.show()